In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Bengaluru_House_Data.csv to Bengaluru_House_Data.csv


In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA

df = pd.read_csv("Bengaluru_House_Data.csv")

# 2. BASIC CLEANING(removed the noise\used .fillna to fill the missing places with other)

df['location'] = df['location'].fillna('other').str.strip()
df = df.dropna(subset=['size', 'total_sqft', 'bath'])

# 3. FEATURE ENGINEERING(data ko edit kiye hain properly to get better accuracy)


# Convert size → bhk
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)

# Convert total_sqft
def convert_sqft(x):
    try:
        x = str(x)
        if '-' in x:
            a, b = x.split('-')
            return (float(a) + float(b)) / 2
        return float(x)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

# Drop useless columns
df = df.drop(columns=['size', 'society'])

# Remove bad data(keeps the data which only above 300 sqft.)
df = df[df['total_sqft'] > 300]
df = df[df['bhk'] <= df['bath'] + 2] #(removes the house if bkh is )

# 4. FIX LOCATION (IMPORTANT)

location_counts = df['location'].value_counts()

df['location'] = df['location'].apply(
    lambda x: 'other' if location_counts.get(x, 0) <= 10 else x
)

# 5. FEATURES & TARGET(we have droped price here so model could not guess direct)

X = df.drop(columns=['price'])   # no leakage
y = df['price']

# 6. ENCODING(converts rest string value into numbers coding)

X = pd.get_dummies(X, drop_first=True)


# 7. HANDLE MISSING VALUES

X = X.fillna(X.mean(numeric_only=True))


# 8. SPLIT DATA

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 9. TRAIN MODEL

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


# 10. EVALUATE

print("Train Score:", model.score(X_train, y_train))
print("Test Score:", model.score(X_test, y_test))


# 11. PREDICTION FUNCTION(made a function to use the input and model to predict prices)

def predict_price():
    print("\nEnter house details:")

    sqft = float(input("Total sqft: "))
    bath = int(input("Bathrooms: "))
    balcony = int(input("Balconies: "))
    bhk = int(input("BHK: "))
    location = input("Location: ").strip()
    availability = input("Availability (Ready To Move / Under Construction): ").strip()
    area_type = input("Area Type: ").strip()

    # Handle unknown locations
    if location not in df['location'].values:
        location = "other"

    new_house = pd.DataFrame({
        'total_sqft': [sqft],
        'bath': [bath],
        'balcony': [balcony],
        'bhk': [bhk],
        'location': [location],
        'availability': [availability],
        'area_type': [area_type]
    })

    # Encode
    new_house = pd.get_dummies(new_house)

    # Match training columns
    new_house = new_house.reindex(columns=X_train.columns, fill_value=0)

    # Predict
    prediction = model.predict(new_house)
    print(f"\n Predicted Price: {prediction[0]:.2f} Lakhs")



# 12. RUN PREDICTION

predict_price()

Train Score: 0.9428063712944688
Test Score: 0.6078013798216426

Enter house details:
Total sqft: 450
Bathrooms: 3
Balconies: 2
BHK: 4
Location: bengaluru
Availability (Ready To Move / Under Construction): Ready to move
Area Type: good

🏠 Predicted Price: 83.28 Lakhs
